# Proyecto EDA | Motor Inteligente de E-Commerce v2

## Qué hace esta versión

La v1 tenía cada estructura en su propia celda y un dashboard que consultaba esas piezas por separado. La v2 hace lo mismo para el usuario (buscar productos, filtrar por precio, recomendar, ver el grafo) y además **une las piezas en un solo flujo**:

| Si llega... | Recorre | Termina en |
|---|---|---|
| Una vista o un carrito | cola, limpieza, heap, almacén | recomendación de 5 productos |
| Una compra | cola, limpieza, heap, almacén | revisión de fraude: permitir, revisar o bloquear |
| Una compra no bloqueada | lo anterior | el grafo de productos comprados juntos |

Todo termina en el **dashboard**, justo después de la traza de tres eventos, con buscador, autocompletado, rango de precios, recomendaciones, simulador de eventos, alertas y grafo.

**Cómo usarlo:** ejecuta todas las celdas en orden (Colab: *Entorno de ejecución > Ejecutar todas*) y baja hasta el dashboard.

---

Sistema integrado para la opción B del curso. Un evento entra por un núcleo común y después sigue **solo la rama que le corresponde**:

1. Ingesta, limpieza, prioridad y almacén.
2. Vista o carrito: recomendación híbrida (grafo de sesión + HNSW).
3. Compra: fraude con Bloom Filter y Count-Min. Solo se bloquea si un conjunto exacto confirma el caso.
4. Compra no bloqueada: grafo de co-compra de la misma sesión, tratado como una orden.
5. Servicio: autocompletado, ranking y panel.

El notebook anterior no se modifica. Aquí las piezas quedan conectadas y cada evento de la demo deja una traza.

`quick` lee 100.000 eventos y mide hasta 100K. `full` lee `MAX_EVENTS` y mide 1K, 10K, 100K, 1M y 10M. Se usa un solo archivo mensual, elegido por nombre. No se descargan los siete meses ni se guardan copias de cada escala.


In [1]:
# Configuración. Cambiar el mes o el tope no crea otro archivo de datos.
DATASET_FILE = "2019-Oct.csv"
MAX_EVENTS = 10_000_000
RUN_MODE = "quick"  # "quick" o "full"
SEED = 42

EVENTOS_A_LEER = 100_000 if RUN_MODE == "quick" else MAX_EVENTS
TAMANOS_BENCH = (
    [1_000, 10_000, 100_000]
    if RUN_MODE == "quick"
    else [1_000, 10_000, 100_000, 1_000_000, 10_000_000]
)
REPETICIONES_QUICK = 7
REPETICIONES_FULL = 100
VECTOR_DIM = 14  # 1 precio + 1 popularidad + 8 categoría + 4 marca
CUOTA_HNSW = 1
CUOTA_GRAFO = 4
UMBRAL_INTENSIDAD = 8
FRAUDE_MODULO = 125  # cerca del 0,8 % de las compras
MIN_INTERACCIONES = 2
MAX_PRODUCTOS_INDICE = 20_000
MAX_PRODUCTOS_SESION = 40

print("Modo:", RUN_MODE)
print("Archivo:", DATASET_FILE)
print("Eventos a leer:", EVENTOS_A_LEER)
print("Semilla:", SEED)


Modo: quick
Archivo: 2019-Oct.csv
Eventos a leer: 100000
Semilla: 42


In [2]:
import hashlib
import heapq
import math
import platform
import random
import time
import tracemalloc
from bisect import bisect_left, bisect_right
from collections import Counter, OrderedDict, defaultdict, deque
from itertools import combinations

import numpy as np
import pandas as pd

rng = random.Random(SEED)
np.random.seed(SEED)


def huella(*partes):
    texto = "|".join(map(str, partes)).encode()
    return int.from_bytes(hashlib.sha256(texto).digest()[:8], "little")


def repeticiones_para(n):
    if RUN_MODE == "quick":
        return REPETICIONES_QUICK
    if n >= 1_000_000:
        # 100 repeticiones de estructuras sobre 1M o 10M no caben en una corrida de clase.
        return 3
    return REPETICIONES_FULL


print("Plataforma:", platform.platform())
print("Semilla:", SEED)


Plataforma: Linux-6.6.122+-x86_64-with-glibc2.39
Semilla: 42


## Carga

Se localiza solo `DATASET_FILE` dentro del dataset de REES46. La lectura es por bloques y se detiene al llegar al tope del modo.


In [3]:
def ubicar_archivo(nombre):
    import os
    candidatos = []
    try:
        import kagglehub
        raiz = kagglehub.dataset_download(
            "mkechinov/ecommerce-behavior-data-from-multi-category-store"
        )
        candidatos.append(os.path.join(raiz, nombre))
    except Exception as exc:
        print("No se usó kagglehub:", exc)
    candidatos.append(os.path.join(os.getcwd(), nombre))
    for ruta in candidatos:
        if os.path.exists(ruta):
            print("Fuente:", ruta)
            return ruta
    raise FileNotFoundError(
        f"No está {nombre}. Cambia DATASET_FILE al mes que quieras usar, "
        "por ejemplo 2019-Nov.csv. No se descargan los siete meses."
    )


def leer_eventos(ruta, max_eventos):
    columnas = [
        "event_time", "event_type", "product_id", "category_id",
        "category_code", "brand", "price", "user_id", "user_session",
    ]
    bloques = []
    restante = max_eventos
    lector = pd.read_csv(ruta, usecols=columnas, chunksize=min(250_000, max_eventos))
    for bloque in lector:
        if len(bloque) > restante:
            bloque = bloque.iloc[:restante]
        bloques.append(bloque)
        restante -= len(bloque)
        if restante <= 0:
            break
    if not bloques:
        raise RuntimeError("El archivo no tiene filas.")
    return pd.concat(bloques, ignore_index=True)


## Núcleo

La limpieza ordena con merge sort. Hasta 100.000 filas el ordenamiento es interno. Por encima de eso, ordena bloques y los mezcla: es el mismo algoritmo, apto para el modo `full`.

Después el lote histórico (60 %) construye el almacén y los modelos. El 40 % restante es el stream vivo: cola, heap y ramas.


In [4]:
def merge_sort(items, clave):
    if len(items) <= 1:
        return list(items)
    tamaño = 1
    ordenados = list(items)
    while tamaño < len(ordenados):
        siguiente = []
        for inicio in range(0, len(ordenados), tamaño * 2):
            izquierda = ordenados[inicio:inicio + tamaño]
            derecha = ordenados[inicio + tamaño:inicio + tamaño * 2]
            i = j = 0
            while i < len(izquierda) and j < len(derecha):
                if clave(izquierda[i]) <= clave(derecha[j]):
                    siguiente.append(izquierda[i])
                    i += 1
                else:
                    siguiente.append(derecha[j])
                    j += 1
            siguiente.extend(izquierda[i:])
            siguiente.extend(derecha[j:])
        ordenados = siguiente
        tamaño *= 2
    return ordenados


def mezcla_k(bloques, clave):
    heap = []
    for b, bloque in enumerate(bloques):
        if bloque:
            heapq.heappush(heap, (clave(bloque[0]), b, 0, bloque[0]))
    salida = []
    while heap:
        _, b, pos, valor = heapq.heappop(heap)
        salida.append(valor)
        pos += 1
        if pos < len(bloques[b]):
            valor = bloques[b][pos]
            heapq.heappush(heap, (clave(valor), b, pos, valor))
    return salida


def ordenar_indices(indices, tiempos, tamano_bloque=100_000):
    def clave(indice):
        return (tiempos[indice], indice)

    if len(indices) <= tamano_bloque:
        return merge_sort(indices, clave)
    bloques = []
    for inicio in range(0, len(indices), tamano_bloque):
        parte = indices[inicio:inicio + tamano_bloque]
        bloques.append(merge_sort(parte, clave))
    return mezcla_k(bloques, clave)


def busqueda_binaria(ordenado, objetivo):
    bajo, alto = 0, len(ordenado) - 1
    while bajo <= alto:
        medio = (bajo + alto) // 2
        if ordenado[medio] == objetivo:
            return medio
        if ordenado[medio] < objetivo:
            bajo = medio + 1
        else:
            alto = medio - 1
    return -1


class CacheLRU:
    def __init__(self, capacidad=256):
        self.capacidad = capacidad
        self.cache = OrderedDict()
        self.aciertos = 0
        self.fallos = 0

    def obtener(self, clave):
        if clave in self.cache:
            self.cache.move_to_end(clave)
            self.aciertos += 1
            return self.cache[clave]
        self.fallos += 1
        return None

    def guardar(self, clave, valor):
        if clave in self.cache:
            self.cache.move_to_end(clave)
        self.cache[clave] = valor
        if len(self.cache) > self.capacidad:
            self.cache.popitem(last=False)


class NodoBPlus:
    def __init__(self, hoja=False):
        self.hoja = hoja
        self.claves = []
        self.hijos = []
        self.siguiente = None


class ArbolBPlus:
    # Árbol en memoria con la interfaz de un índice de disco: puntual y por rango.
    def __init__(self, orden=32):
        self.orden = orden
        self.raiz = NodoBPlus(hoja=True)

    def insertar(self, clave, valor):
        clave = float(clave)
        promovido = self._insertar(self.raiz, clave, valor)
        if promovido is not None:
            separador, derecho = promovido
            raiz = NodoBPlus(hoja=False)
            raiz.claves = [separador]
            raiz.hijos = [self.raiz, derecho]
            self.raiz = raiz

    def _insertar(self, nodo, clave, valor):
        if nodo.hoja:
            i = bisect_left(nodo.claves, clave)
            if i < len(nodo.claves) and nodo.claves[i] == clave:
                nodo.hijos[i].append(valor)
            else:
                nodo.claves.insert(i, clave)
                nodo.hijos.insert(i, [valor])
            if len(nodo.claves) <= self.orden:
                return None
            return self._dividir_hoja(nodo)
        i = bisect_right(nodo.claves, clave)
        promovido = self._insertar(nodo.hijos[i], clave, valor)
        if promovido is None:
            return None
        separador, derecho = promovido
        nodo.claves.insert(i, separador)
        nodo.hijos.insert(i + 1, derecho)
        if len(nodo.claves) <= self.orden:
            return None
        return self._dividir_interno(nodo)

    def _dividir_hoja(self, nodo):
        medio = len(nodo.claves) // 2
        nuevo = NodoBPlus(hoja=True)
        nuevo.claves = nodo.claves[medio:]
        nuevo.hijos = nodo.hijos[medio:]
        nodo.claves = nodo.claves[:medio]
        nodo.hijos = nodo.hijos[:medio]
        nuevo.siguiente = nodo.siguiente
        nodo.siguiente = nuevo
        return nuevo.claves[0], nuevo

    def _dividir_interno(self, nodo):
        medio = len(nodo.claves) // 2
        separador = nodo.claves[medio]
        nuevo = NodoBPlus(hoja=False)
        nuevo.claves = nodo.claves[medio + 1:]
        nuevo.hijos = nodo.hijos[medio + 1:]
        nodo.claves = nodo.claves[:medio]
        nodo.hijos = nodo.hijos[:medio + 1]
        return separador, nuevo

    def buscar_rango(self, minimo, maximo):
        nodo = self.raiz
        while not nodo.hoja:
            nodo = nodo.hijos[bisect_right(nodo.claves, minimo)]
        salida = []
        while nodo is not None:
            for clave, valores in zip(nodo.claves, nodo.hijos):
                if minimo <= clave <= maximo:
                    salida.extend(valores)
                elif clave > maximo:
                    return salida
            nodo = nodo.siguiente
        return salida


class Almacen:
    def __init__(self):
        self.catalogo = {}
        self.lru = CacheLRU(256)
        self.precios = ArbolBPlus()
        self.interacciones = {}
        self.pares = 0

    def actualizar(self, evento):
        producto = int(evento["product_id"])
        usuario = int(evento["user_id"])
        if producto not in self.catalogo:
            self.catalogo[producto] = {
                "categoria": evento["category_code"],
                "marca": evento["brand"],
                "precio": float(evento["price"]),
                "disponible": bool(evento["disponible"]),
                "popularidad": 0,
            }
            self.precios.insertar(float(evento["price"]), producto)
        self.catalogo[producto]["popularidad"] += 1
        por_usuario = self.interacciones.setdefault(usuario, {})
        if producto not in por_usuario:
            self.pares += 1
        por_usuario[producto] = por_usuario.get(producto, 0) + 1
        return self.consultar(producto)

    def consultar(self, producto):
        en_cache = self.lru.obtener(producto)
        if en_cache is not None:
            return en_cache, "CACHE"
        item = self.catalogo.get(producto)
        if item is None:
            return None, "NO ENCONTRADO"
        self.lru.guardar(producto, item)
        return item, "HASH"

    def memoria_dispersa(self):
        usuarios = len(self.interacciones)
        productos = len(self.catalogo)
        densa = usuarios * productos * 8
        dispersa = self.pares * 32
        return {
            "usuarios": usuarios,
            "productos": productos,
            "pares": self.pares,
            "densa_mb": densa / (1024 ** 2),
            "dispersa_mb": dispersa / (1024 ** 2),
        }


In [5]:
class BloomFilter:
    def __init__(self, capacidad, error=0.01):
        capacidad = max(1, int(capacidad))
        self.m = max(8, math.ceil(-(capacidad * math.log(error)) / (math.log(2) ** 2)))
        self.k = max(1, round((self.m / capacidad) * math.log(2)))
        self.bits = bytearray(math.ceil(self.m / 8))

    def _posiciones(self, elemento):
        texto = str(elemento).encode()
        for i in range(self.k):
            digest = hashlib.sha256(i.to_bytes(2, "little") + texto).digest()
            yield int.from_bytes(digest[:4], "little") % self.m

    def agregar(self, elemento):
        for pos in self._posiciones(elemento):
            self.bits[pos // 8] |= 1 << (pos % 8)

    def contiene(self, elemento):
        return all(self.bits[pos // 8] & (1 << (pos % 8)) for pos in self._posiciones(elemento))


class CountMinSketch:
    def __init__(self, ancho=4096, profundidad=5):
        self.ancho = ancho
        self.profundidad = profundidad
        self.tabla = np.zeros((profundidad, ancho), dtype=np.int64)

    def _pos(self, elemento, i):
        digest = hashlib.sha256(f"{i}|{elemento}".encode()).digest()
        return int.from_bytes(digest[:4], "little") % self.ancho

    def agregar(self, elemento, veces=1):
        for i in range(self.profundidad):
            self.tabla[i, self._pos(elemento, i)] += veces

    def estimar(self, elemento):
        return int(min(self.tabla[i, self._pos(elemento, i)] for i in range(self.profundidad)))


class Trie:
    def __init__(self):
        self.raiz = {}

    def insertar(self, texto, valor):
        nodo = self.raiz
        for caracter in str(texto).lower():
            nodo = nodo.setdefault(caracter, {})
        nodo.setdefault("_vals", []).append(valor)

    def completar(self, prefijo, limite=5):
        nodo = self.raiz
        for caracter in str(prefijo).lower():
            if caracter not in nodo:
                return []
            nodo = nodo[caracter]
        cola = deque([nodo])
        hallados = []
        while cola and len(hallados) < limite:
            actual = cola.popleft()
            hallados.extend(actual.get("_vals", []))
            for caracter in sorted(clave for clave in actual if clave != "_vals"):
                cola.append(actual[caracter])
        return hallados[:limite]


def agregar_arista(grafo, a, b, peso=1):
    if a == b:
        return
    grafo.setdefault(a, {})
    grafo.setdefault(b, {})
    grafo[a][b] = grafo[a].get(b, 0) + peso
    grafo[b][a] = grafo[b].get(a, 0) + peso


def construir_grafo_sesion(df):
    grafo = {}
    sesiones = df.groupby("user_session", sort=False)["product_id"]
    for _, productos in sesiones:
        unicos = list(dict.fromkeys(int(p) for p in productos))[:MAX_PRODUCTOS_SESION]
        for a, b in combinations(unicos, 2):
            agregar_arista(grafo, a, b)
    return grafo


def candidatos_grafo(vistos, grafo, k=20):
    puntaje = {}
    vistos = set(vistos)
    for producto in vistos:
        for vecino, peso in grafo.get(producto, {}).items():
            if vecino in vistos:
                continue
            puntaje[vecino] = puntaje.get(vecino, 0) + peso
    orden = sorted(puntaje.items(), key=lambda item: (-item[1], item[0]))
    return [producto for producto, _ in orden[:k]]


def vector_producto(precio, popularidad, categoria, marca, escala):
    precio_n = math.log1p(max(precio, 0.0)) / escala["precio"]
    pop_n = math.log1p(max(popularidad, 0)) / escala["popularidad"]
    vector = [precio_n, pop_n] + [0.0] * 12
    vector[2 + (huella("cat", categoria) % 8)] = 1.0
    vector[10 + (huella("marca", marca) % 4)] = 1.0
    return vector


class IndiceVectorial:
    def __init__(self, vectores, ids):
        self.vectores = np.asarray(vectores, dtype=np.float32)
        self.ids = [int(i) for i in ids]
        self.backend = "knn_exacto"
        self.index = None
        if len(self.ids) == 0:
            return
        try:
            import hnswlib
            indice = hnswlib.Index(space="l2", dim=self.vectores.shape[1])
            indice.init_index(max_elements=len(self.ids), ef_construction=80, M=16)
            indice.add_items(self.vectores, np.arange(len(self.ids)))
            indice.set_ef(40)
            self.index = indice
            self.backend = "hnswlib"
        except Exception:
            self.backend = "knn_exacto"

    def buscar(self, consulta, k):
        if len(self.ids) == 0:
            return [], []
        k = min(k, len(self.ids))
        consulta = np.asarray(consulta, dtype=np.float32).reshape(1, -1)
        if self.index is not None:
            etiquetas, distancias = self.index.knn_query(consulta, k=k)
            return [self.ids[i] for i in etiquetas[0]], distancias[0]
        distancias = np.linalg.norm(self.vectores - consulta[0], axis=1)
        orden = np.argsort(distancias)[:k]
        return [self.ids[i] for i in orden], distancias[orden]


def knn_exacto(vectores, consulta, k):
    distancias = np.linalg.norm(vectores - np.asarray(consulta, dtype=np.float32), axis=1)
    orden = np.argsort(distancias)[:k]
    return orden, distancias[orden]


def construir_indice_productos(df, almacen):
    conteo = Counter()
    for usuario, productos in almacen.interacciones.items():
        for producto in productos:
            conteo[producto] += 1
    elegibles = [p for p, c in conteo.items() if c >= MIN_INTERACCIONES and p in almacen.catalogo]
    elegibles.sort(key=lambda p: conteo[p], reverse=True)
    elegibles = elegibles[:MAX_PRODUCTOS_INDICE]
    if not elegibles:
        elegibles = list(almacen.catalogo)[:MAX_PRODUCTOS_INDICE]
    precio_max = max(math.log1p(max(almacen.catalogo[p]["precio"], 0.0)) for p in elegibles)
    pop_max = max(math.log1p(almacen.catalogo[p]["popularidad"]) for p in elegibles)
    escala = {
        "precio": precio_max if precio_max else 1.0,
        "popularidad": pop_max if pop_max else 1.0,
    }
    vectores = []
    for producto in elegibles:
        item = almacen.catalogo[producto]
        vectores.append(vector_producto(item["precio"], item["popularidad"], item["categoria"], item["marca"], escala))
    return {
        "ids": elegibles,
        "vectores": np.asarray(vectores, dtype=np.float32) if vectores else np.zeros((0, VECTOR_DIM), dtype=np.float32),
        "escala": escala,
        "indice": IndiceVectorial(vectores, elegibles) if vectores else IndiceVectorial(np.zeros((0, VECTOR_DIM)), []),
        "populares": [p for p, _ in Counter(conteo).most_common(20)],
    }


def candidatos_vector(vistos, modelo, k=20):
    posiciones = []
    id_a_pos = {producto: i for i, producto in enumerate(modelo["ids"])}
    for producto in vistos:
        if producto in id_a_pos:
            posiciones.append(id_a_pos[producto])
    if not posiciones:
        return list(modelo["populares"])
    consulta = modelo["vectores"][posiciones].mean(axis=0)
    ids, _ = modelo["indice"].buscar(consulta, k + len(posiciones))
    vistos = set(vistos)
    return [producto for producto in ids if producto not in vistos][:k]


def hibrido(vistos, grafo, modelo, k=5):
    de_vector = candidatos_vector(vistos, modelo, k=20)
    de_grafo = candidatos_grafo(vistos, grafo, k=20)
    salida = []
    vistos = set(vistos)

    def tomar(lista, cupo):
        tomados = 0
        for producto in lista:
            if producto in vistos or producto in salida:
                continue
            salida.append(producto)
            tomados += 1
            if tomados >= cupo or len(salida) >= k:
                break

    tomar(de_vector, CUOTA_HNSW)
    tomar(de_grafo, CUOTA_GRAFO)
    if len(salida) < k:
        tomar(de_vector + de_grafo + modelo["populares"], k)
    return salida[:k]


In [6]:
def decidir_fraude(evento, bloom, cms, exactos):
    motivos = []
    if bloom.contiene(evento["device_id"]):
        motivos.append("dispositivo")
    if bloom.contiene(evento["payment_id"]):
        motivos.append("pago")
    clave = f"{int(evento['user_id'])}|{evento['hora']}"
    intensidad = cms.estimar(clave)
    if intensidad >= UMBRAL_INTENSIDAD:
        motivos.append(f"intensidad:{intensidad}")
    confirmado = bool(evento["fraude_confirmado"]) or evento["device_id"] in exactos or evento["payment_id"] in exactos
    if confirmado:
        return "bloquear", motivos or ["confirmado"]
    if motivos:
        return "revisar", motivos
    return "permitir", []


def construir_fraude(df, corte):
    historico = df.iloc[:corte]
    confirmados = set()
    vigilados = set()
    cms = CountMinSketch()
    # La ventana es el histórico ya cerrado. No cuenta compras futuras de la misma hora.
    for fila in historico.itertuples(index=False):
        if fila.event_type != "purchase":
            continue
        cms.agregar(f"{int(fila.user_id)}|{fila.hora}")
        if fila.fraude_confirmado:
            confirmados.add(fila.device_id)
            confirmados.add(fila.payment_id)
        elif huella("vigia", fila.device_id) % 50 == 0:
            vigilados.add(fila.device_id)
    miembros = confirmados | vigilados
    bloom = BloomFilter(max(len(miembros), 1024), error=0.01)
    for miembro in miembros:
        bloom.agregar(miembro)
    return {"bloom": bloom, "cms": cms, "exactos": confirmados, "vigilados": len(miembros)}


def construir_cocompra(df, indices):
    try:
        import networkx as nx
    except ImportError:
        nx = None
    compras = df.loc[list(indices)]
    if nx is None:
        return {"nx": None, "grafo": None, "por_sesion": {}, "nodos": 0, "aristas": 0}
    grafo = nx.Graph()
    por_sesion = defaultdict(list)
    for indice, fila in compras.iterrows():
        producto = int(fila["product_id"])
        grafo.add_node(producto)
        por_sesion[fila["user_session"]].append((indice, producto, fila["event_time"]))
    for pares in por_sesion.values():
        pares.sort(key=lambda item: item[2])
        vistos = []
        for _, producto, _ in pares:
            for otro in vistos:
                if otro == producto:
                    continue
                if grafo.has_edge(otro, producto):
                    grafo[otro][producto]["weight"] += 1
                else:
                    grafo.add_edge(otro, producto, weight=1)
            vistos.append(producto)
    return {
        "nx": nx,
        "grafo": grafo,
        "por_sesion": por_sesion,
        "nodos": grafo.number_of_nodes(),
        "aristas": grafo.number_of_edges(),
    }


def unir_compra(estado, indice, producto, sesion, momento):
    grafo = estado["grafo"]
    if grafo is None:
        return 0
    pares = estado["por_sesion"].setdefault(sesion, [])
    previos = [otro for j, otro, cuando in pares if j != indice and cuando < momento]
    grafo.add_node(producto)
    nuevas = 0
    for otro in dict.fromkeys(previos):
        if otro == producto:
            continue
        if grafo.has_edge(otro, producto):
            grafo[otro][producto]["weight"] += 1
        else:
            grafo.add_edge(otro, producto, weight=1)
            nuevas += 1
    pares.append((indice, producto, momento))
    return nuevas


def pagerank_propio(grafo, damping=0.85, iters=30):
    nodos = list(grafo.nodes)
    n = len(nodos)
    if n == 0:
        return {}
    indice = {nodo: i for i, nodo in enumerate(nodos)}
    vecinos = []
    salida = []
    for nodo in nodos:
        pares = []
        total = 0.0
        for _, vecino, datos in grafo.edges(nodo, data=True):
            peso = float(datos.get("weight", 1))
            pares.append((indice[vecino], peso))
            total += peso
        vecinos.append(pares)
        salida.append(total)
    rank = [1.0 / n] * n
    for _ in range(iters):
        nuevo = [(1 - damping) / n] * n
        colgado = 0.0
        for i, total in enumerate(salida):
            if total == 0:
                colgado += rank[i]
        if colgado:
            reparto = damping * colgado / n
            nuevo = [valor + reparto for valor in nuevo]
        for i, pares in enumerate(vecinos):
            if salida[i] == 0:
                continue
            for destino, peso in pares:
                nuevo[destino] += damping * rank[i] * (peso / salida[i])
        rank = nuevo
    return {nodo: rank[indice[nodo]] for nodo in nodos}


def resumen_grafo(estado):
    grafo = estado["grafo"]
    nx = estado["nx"]
    if grafo is None or grafo.number_of_nodes() == 0:
        return {"nodos": 0, "aristas": 0, "pagerank": [], "grado": [], "comunidades": 0, "modularidad": None}
    grado = sorted(grafo.degree(weight="weight"), key=lambda item: item[1], reverse=True)[:10]
    if grafo.number_of_edges() == 0:
        return {
            "nodos": grafo.number_of_nodes(),
            "aristas": 0,
            "pagerank": grado,
            "grado": grado,
            "comunidades": grafo.number_of_nodes(),
            "modularidad": None,
        }
    puntajes = sorted(pagerank_propio(grafo).items(), key=lambda item: item[1], reverse=True)[:10]
    comunidades = []
    modularidad = None
    if nx is not None:
        try:
            comunidades = list(nx.community.louvain_communities(grafo, weight="weight", seed=SEED))
            modularidad = float(nx.community.modularity(grafo, comunidades, weight="weight"))
        except Exception:
            comunidades = list(nx.connected_components(grafo))
    return {
        "nodos": grafo.number_of_nodes(),
        "aristas": grafo.number_of_edges(),
        "pagerank": puntajes,
        "grado": grado,
        "comunidades": len(comunidades),
        "modularidad": modularidad,
        "particion": comunidades,
    }


In [7]:
def preparar_eventos(df, semilla):
    df = df.copy()
    df = df.dropna(subset=["event_time", "event_type", "product_id", "price", "user_id", "user_session"])
    df["category_code"] = df["category_code"].fillna("unknown").astype(str).replace("", "unknown")
    df["brand"] = df["brand"].fillna("unknown").astype(str).replace("", "unknown")
    df["event_time"] = pd.to_datetime(df["event_time"], utc=True, format="mixed")
    df["product_id"] = df["product_id"].astype("int64")
    df["user_id"] = df["user_id"].astype("int64")
    df["price"] = df["price"].astype(float)
    antes = len(df)
    df = df.drop_duplicates(["event_time", "event_type", "product_id", "user_id", "user_session"])
    duplicados = antes - len(df)
    df = df.reset_index(drop=True)
    tiempos = df["event_time"].astype("int64").to_numpy()
    orden = ordenar_indices(list(range(len(df))), tiempos)
    df = df.iloc[orden].reset_index(drop=True)
    dispositivos = np.empty(len(df), dtype=object)
    pagos = np.empty(len(df), dtype=object)
    entregas = np.empty(len(df), dtype=object)
    fraude = np.zeros(len(df), dtype=bool)
    disponible = np.ones(len(df), dtype=bool)
    prioridad = np.empty(len(df), dtype=np.int16)
    etiqueta = np.empty(len(df), dtype=object)
    nombres_dispositivo = ["mobile", "desktop", "tablet"]
    nombres_pago = ["tarjeta", "billetera", "efectivo"]
    for i, (usuario, producto, tipo) in enumerate(zip(df["user_id"], df["product_id"], df["event_type"])):
        h = huella(semilla, int(usuario), int(producto))
        dispositivos[i] = f"dev-{int(usuario)}-{h % 2}"
        pagos[i] = f"pay-{int(usuario)}-{h % 3}"
        bucket = h % 5
        entregas[i] = "Express" if bucket == 0 else "Programada" if bucket == 4 else "Estandar"
        disponible[i] = (h % 20) != 0
        es_fraude = tipo == "purchase" and (h % FRAUDE_MODULO == 0)
        fraude[i] = es_fraude
        if tipo == "purchase" and es_fraude:
            prioridad[i], etiqueta[i] = 1, "Fraude"
        elif tipo == "purchase" and entregas[i] == "Express":
            prioridad[i], etiqueta[i] = 2, "Express"
        elif tipo == "purchase" and entregas[i] == "Estandar":
            prioridad[i], etiqueta[i] = 3, "Estandar"
        elif tipo == "purchase":
            prioridad[i], etiqueta[i] = 4, "Programada"
        elif tipo == "cart":
            prioridad[i], etiqueta[i] = 5, "Carrito"
        else:
            prioridad[i], etiqueta[i] = 6, "Vista"
    df["device_id"] = dispositivos
    df["payment_id"] = pagos
    df["entrega"] = entregas
    df["fraude_confirmado"] = fraude
    df["disponible"] = disponible
    df["prioridad"] = prioridad
    df["etiqueta_prioridad"] = etiqueta
    df["hora"] = df["event_time"].dt.floor("h").astype(str)
    df["evento_id"] = [f"E{i:08d}" for i in range(len(df))]
    return df, duplicados


def cortes_temporales(n):
    train = max(1, int(n * 0.60))
    val = max(train + 1, int(n * 0.70))
    if val >= n and n > train:
        val = n - 1
    return train, val


def actualizar_almacen(almacen, df, indices):
    for indice in indices:
        almacen.actualizar(df.iloc[indice])


def ejecutar_pipeline(df_crudo):
    df, duplicados = preparar_eventos(df_crudo, SEED)
    n = len(df)
    corte_train, corte_val = cortes_temporales(n)
    almacen = Almacen()
    actualizar_almacen(almacen, df, range(corte_train))
    grafo_sesion = construir_grafo_sesion(df.iloc[:corte_train])
    modelo = construir_indice_productos(df.iloc[:corte_train], almacen)
    fraude = construir_fraude(df, corte_train)
    compras_train = df.index[:corte_train][df.iloc[:corte_train]["event_type"].to_numpy() == "purchase"]
    cocompra = construir_cocompra(df, compras_train)
    trie = Trie()
    for producto, item in almacen.catalogo.items():
        trie.insertar(f"{item['categoria']} {item['marca']}", producto)
        trie.insertar(f"{item['marca']} {item['categoria']}", producto)

    productos_np = df["product_id"].to_numpy()
    prefijos = {}
    indices_sesion = defaultdict(list)
    for indice, sesion in enumerate(df["user_session"].tolist()):
        indices_sesion[sesion].append(indice)
    for indices in indices_sesion.values():
        vistos = []
        for indice in indices:
            prefijos[indice] = list(vistos)
            producto = int(productos_np[indice])
            if producto not in vistos:
                vistos.append(producto)

    cola = deque(range(corte_train, n))
    heap = []
    for orden, indice in enumerate(cola):
        heapq.heappush(heap, (int(df.at[indice, "prioridad"]), orden, indice))
    recibidos = len(cola)
    cola.clear()

    trazas = []
    por_indice = {}
    permitidas = []
    primeros = []
    detallado = n <= 100_000
    while heap:
        prioridad, _, indice = heapq.heappop(heap)
        fila = df.iloc[indice]
        item, origen = almacen.actualizar(fila)
        tipo = fila["event_type"]
        if len(primeros) < 10:
            primeros.append((fila["evento_id"], int(prioridad), fila["etiqueta_prioridad"], int(fila["product_id"])))
        if tipo in ("view", "cart"):
            vistos = prefijos[indice] + [int(fila["product_id"])]
            recomendaciones = hibrido(vistos, grafo_sesion, modelo) if detallado else []
            traza = {
                "evento_id": fila["evento_id"],
                "indice": indice,
                "tipo": tipo,
                "prioridad": int(prioridad),
                "etiqueta": fila["etiqueta_prioridad"],
                "rama": "recomendacion",
                "decision": "consultar",
                "motivos": "",
                "recomendaciones": recomendaciones,
                "actualizo_cocompra": False,
                "aristas_nuevas": 0,
                "origen_lookup": origen,
                "producto": int(fila["product_id"]),
                "texto": f"{fila['category_code']} {fila['brand']}",
            }
        else:
            decision, motivos = decidir_fraude(fila, fraude["bloom"], fraude["cms"], fraude["exactos"])
            traza = {
                "evento_id": fila["evento_id"],
                "indice": indice,
                "tipo": tipo,
                "prioridad": int(prioridad),
                "etiqueta": fila["etiqueta_prioridad"],
                "rama": "fraude",
                "decision": decision,
                "motivos": ",".join(motivos),
                "recomendaciones": [],
                "actualizo_cocompra": False,
                "aristas_nuevas": 0,
                "origen_lookup": origen,
                "producto": int(fila["product_id"]),
                "texto": f"{fila['category_code']} {fila['brand']}",
            }
            if decision != "bloquear":
                permitidas.append(indice)
                traza["rama_final"] = "cocompra"
            else:
                traza["rama_final"] = "fraude"
        trazas.append(traza)
        por_indice[indice] = traza

    tiempos = df["event_time"]
    for indice in sorted(permitidas, key=lambda i: tiempos.iloc[i]):
        fila = df.iloc[indice]
        nuevas = unir_compra(
            cocompra,
            indice,
            int(fila["product_id"]),
            fila["user_session"],
            fila["event_time"],
        )
        por_indice[indice]["actualizo_cocompra"] = True
        por_indice[indice]["aristas_nuevas"] = nuevas

    for traza in elegir_ejemplos(trazas):
        if traza["tipo"] in ("view", "cart") and not traza["recomendaciones"]:
            vistos = prefijos[traza["indice"]] + [traza["producto"]]
            traza["recomendaciones"] = hibrido(vistos, grafo_sesion, modelo)

    ids = sorted(almacen.catalogo)
    buscado = ids[len(ids) // 2]
    posicion = busqueda_binaria(ids, buscado)
    precios = [almacen.catalogo[p]["precio"] for p in ids[:5000]]
    if precios:
        bajo, alto = np.percentile(precios, [40, 60])
        en_rango = almacen.precios.buscar_rango(float(bajo), float(alto))
    else:
        bajo = alto = 0
        en_rango = []
    knn_ok = validar_kd(almacen)
    resumen = resumen_grafo(cocompra)
    panel = {
        "eventos_stream": len(trazas),
        "consultas": sum(t["rama"] == "recomendacion" for t in trazas),
        "ventas": sum(t["tipo"] == "purchase" and t["decision"] != "bloquear" for t in trazas),
        "alertas": sum(t["decision"] in ("revisar", "bloquear") for t in trazas),
        "bloqueos": sum(t["decision"] == "bloquear" for t in trazas),
        "revisiones": sum(t["decision"] == "revisar" for t in trazas),
    }
    return {
        "df": df,
        "duplicados": duplicados,
        "corte_train": corte_train,
        "corte_val": corte_val,
        "almacen": almacen,
        "grafo_sesion": grafo_sesion,
        "modelo": modelo,
        "fraude": fraude,
        "cocompra": cocompra,
        "resumen_grafo": resumen,
        "trie": trie,
        "trazas": trazas,
        "primeros_heap": primeros,
        "recibidos_fifo": recibidos,
        "panel": panel,
        "busqueda": {"producto": buscado, "posicion": posicion, "rango": (float(bajo), float(alto)), "en_rango": len(en_rango)},
        "knn_ok": knn_ok,
    }


def elegir_ejemplos(trazas):
    vista = next((t for t in trazas if t["tipo"] == "view"), None)
    bloqueada = next((t for t in trazas if t["decision"] == "bloquear"), None)
    permitida = next((t for t in trazas if t["tipo"] == "purchase" and t["decision"] == "permitir"), None)
    return [t for t in (vista, bloqueada, permitida) if t is not None]


class NodoKD:
    def __init__(self, punto, indice, eje):
        self.punto = punto
        self.indice = indice
        self.eje = eje
        self.izq = None
        self.der = None


def construir_kd(puntos, profundidad=0):
    if not puntos:
        return None
    eje = profundidad % 2
    puntos = sorted(puntos, key=lambda item: item[0][eje])
    medio = len(puntos) // 2
    nodo = NodoKD(puntos[medio][0], puntos[medio][1], eje)
    nodo.izq = construir_kd(puntos[:medio], profundidad + 1)
    nodo.der = construir_kd(puntos[medio + 1:], profundidad + 1)
    return nodo


def buscar_kd(nodo, consulta, k, heap):
    if nodo is None:
        return
    dist = (nodo.punto[0] - consulta[0]) ** 2 + (nodo.punto[1] - consulta[1]) ** 2
    heapq.heappush(heap, (-dist, nodo.indice))
    if len(heap) > k:
        heapq.heappop(heap)
    diff = consulta[nodo.eje] - nodo.punto[nodo.eje]
    primero, segundo = (nodo.izq, nodo.der) if diff < 0 else (nodo.der, nodo.izq)
    buscar_kd(primero, consulta, k, heap)
    if len(heap) < k or diff * diff < -heap[0][0]:
        buscar_kd(segundo, consulta, k, heap)


def validar_kd(almacen):
    items = list(almacen.catalogo.items())[:3000]
    if len(items) < 5:
        return True
    puntos = []
    for producto, item in items:
        puntos.append(((item["precio"], float(item["popularidad"])), producto))
    arbol = construir_kd(puntos)
    consulta = puntos[0][0]
    heap = []
    buscar_kd(arbol, consulta, 5, heap)
    brute = sorted(
        (((p[0] - consulta[0]) ** 2 + (p[1] - consulta[1]) ** 2, producto) for p, producto in puntos)
    )[:5]
    obtenidos = sorted(dist for dist, _ in ((-d, i) for d, i in heap))
    esperados = sorted(dist for dist, _ in brute)
    return np.allclose(obtenidos, esperados)


def evaluar_recomendacion(resultado):
    df = resultado["df"]
    corte_train = resultado["corte_train"]
    corte_val = resultado["corte_val"]
    catalogo = set(resultado["modelo"]["ids"]) or set(resultado["almacen"].catalogo)

    def casos(inicio, fin):
        ventana = df.iloc[inicio:fin]
        sesiones = ventana["user_session"].unique()
        salida = []
        for sesion in sesiones:
            if len(salida) >= 300:
                break
            historia = df[(df["user_session"] == sesion) & (df.index < inicio)]
            vistos = list(dict.fromkeys(int(p) for p in historia["product_id"]))
            if not vistos:
                continue
            objetivos = [
                int(p)
                for p in dict.fromkeys(ventana.loc[ventana["user_session"] == sesion, "product_id"])
                if int(p) not in vistos and int(p) in catalogo
            ]
            if objetivos:
                salida.append((vistos, objetivos))
        return salida

    def puntuar(casos_eval):
        filas = []
        for nombre, funcion in (
            ("Grafo de sesión", lambda vistos: candidatos_grafo(vistos, resultado["grafo_sesion"], k=5)),
            ("HNSW", lambda vistos: candidatos_vector(vistos, resultado["modelo"], k=5)),
            ("Híbrido 1+4", lambda vistos: hibrido(vistos, resultado["grafo_sesion"], resultado["modelo"])),
        ):
            hits, precisiones, recalls, tiempos = [], [], [], []
            for vistos, objetivos in casos_eval:
                t0 = time.perf_counter()
                recomendados = funcion(vistos)
                tiempos.append((time.perf_counter() - t0) * 1000)
                aciertos = len(set(recomendados) & set(objetivos))
                hits.append(1.0 if aciertos else 0.0)
                precisiones.append(aciertos / 5)
                recalls.append(aciertos / len(objetivos))
            filas.append({
                "metodo": nombre,
                "sesiones": len(casos_eval),
                "hit_rate_5": float(np.mean(hits)) if hits else 0.0,
                "precision_5": float(np.mean(precisiones)) if precisiones else 0.0,
                "recall_5": float(np.mean(recalls)) if recalls else 0.0,
                "latencia_mediana_ms": float(np.median(tiempos)) if tiempos else 0.0,
            })
        return pd.DataFrame(filas)

    validacion = puntuar(casos(corte_train, corte_val))
    prueba = puntuar(casos(corte_val, len(df)))
    return validacion, prueba


## Ejecución

Esta celda descarga o abre un solo mes y recorre el pipeline. En `quick` son 100.000 eventos. Para la escala del curso cambia `RUN_MODE` a `full`.


In [8]:
try:
    import hnswlib  # noqa: F401
except ImportError:
    import subprocess
    import sys
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "hnswlib"])
    except Exception as exc:
        print("Sin hnswlib. El índice vectorial usa KNN exacto como respaldo:", exc)

ruta = ubicar_archivo(DATASET_FILE)
crudo = leer_eventos(ruta, EVENTOS_A_LEER)
resultado = ejecutar_pipeline(crudo)
df = resultado["df"]
print("Eventos leídos:", len(crudo))
print("Eventos limpios:", len(df))
print("Duplicados exactos eliminados:", resultado["duplicados"])
print("Corte entrenamiento / validación:", resultado["corte_train"], resultado["corte_val"])
print("Stream vivo:", resultado["recibidos_fifo"])
print("Índice vectorial:", resultado["modelo"]["indice"].backend, "dim", VECTOR_DIM, "productos", len(resultado["modelo"]["ids"]))
print("Co-compra al cierre: nodos", resultado["resumen_grafo"]["nodos"], "aristas", resultado["resumen_grafo"]["aristas"])
print("Panel:", resultado["panel"])
print("Primeros eventos del heap:")
for evento_id, prioridad, etiqueta, producto in resultado["primeros_heap"]:
    print(f"  {evento_id} | prioridad {prioridad} | {etiqueta} | producto {producto}")


100%|██████████| 4.29G/4.29G [00:47<00:00, 97.7MB/s]

Extracting files...


Fuente: /root/.cache/kagglehub/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store/versions/8/2019-Oct.csv
Eventos leídos: 100000
Eventos limpios: 99983
Duplicados exactos eliminados: 17
Corte entrenamiento / validación: 59989 69988
Stream vivo: 39994
Índice vectorial: hnswlib dim 14 productos 5452
Co-compra al cierre: nodos 840 aristas 167
Panel: {'eventos_stream': 39994, 'consultas': 39273, 'ventas': 714, 'alertas': 7, 'bloqueos': 7, 'revisiones': 0}
Primeros eventos del heap:
  E00062426 | prioridad 1 | Fraude | producto 41900000
  E00062527 | prioridad 1 | Fraude | producto 15800120
  E00080165 | prioridad 1 | Fraude | producto 1005106
  E00083914 | prioridad 1 | Fraude | producto 1005106
  E00091998 | prioridad 1 | Fraude | producto 2701419
  E00095316 | prioridad 1 | Fraude | producto 1005106
  E00060242 | prioridad 2 | Express | producto 29501823
  E00061140 | prioridad 2 | Express | producto 26403427
  E00061717 | prioridad 2 | Express | producto 1005062
  E000

## Traza de tres eventos

Una vista solo se recomienda. Una compra confirmada se bloquea y no entra al grafo. Una compra permitida actualiza la co-compra. Las ramas que no corresponden quedan marcadas como omitidas.


In [9]:
def traza_completa(resultado, traza):
    df = resultado["df"]
    fila = df.iloc[traza["indice"]]
    pasos = [
        ("0 Ingesta", "deque", traza["evento_id"], "encolado en el stream vivo"),
        ("1 Limpieza", "merge sort + dedup", traza["evento_id"], "conservado y ordenado por tiempo"),
        ("2 Prioridad", "min-heap", traza["etiqueta"], f"prioridad {traza['prioridad']}"),
        ("3 Almacén", "hash + LRU", str(traza["producto"]), traza["origen_lookup"]),
        ("4 Índice", "B+Tree", "precio", "el árbol se actualiza al ver el producto"),
        ("5 KNN exacto", "KD-Tree", "precio y popularidad", "línea base ya contrastada con fuerza bruta"),
    ]
    if traza["tipo"] in ("view", "cart"):
        pasos.append(("6 Fraude", "Bloom + Count-Min", traza["tipo"], "omitida: no es un pago"))
        pasos.append(("7 Recomendación", f"híbrido 1+4 / {resultado['modelo']['indice'].backend}", traza["texto"], ",".join(map(str, traza["recomendaciones"])) or "sin candidatos"))
        pasos.append(("8 Co-compra", "grafo de orden", traza["tipo"], "omitida: no es una compra confirmada"))
    else:
        pasos.append(("6 Fraude", "Bloom + Count-Min + conjunto exacto", traza["motivos"] or "sin señal", traza["decision"]))
        pasos.append(("7 Recomendación", "híbrido", traza["tipo"], "omitida: la compra no pide una sugerencia"))
        if traza["actualizo_cocompra"]:
            detalle = f"aristas nuevas {traza['aristas_nuevas']}"
        else:
            detalle = "no entra: la compra fue bloqueada"
        pasos.append(("8 Co-compra", "grafo de la sesión", str(traza["producto"]), detalle))
    sugerencias = resultado["trie"].completar(str(fila["category_code"]).split(".")[0], limite=5)
    pasos.append(("9 Servicio", "Trie + panel", str(fila["category_code"]).split(".")[0], ",".join(map(str, sugerencias[:5]))))
    return pasos


ejemplos = elegir_ejemplos(resultado["trazas"])
if not ejemplos:
    print("El stream no tuvo eventos para la demo.")
for traza in ejemplos:
    print("=" * 72)
    print(traza["evento_id"], "|", traza["tipo"], "| producto", traza["producto"], "|", traza["decision"])
    for etapa, estructura, entrada, salida in traza_completa(resultado, traza):
        print(f"{etapa:16} | {estructura:32} | {entrada} -> {salida}")

print("\nPanel de servicio")
for clave, valor in resultado["panel"].items():
    print(f"  {clave}: {valor}")
busqueda = resultado["busqueda"]
print(
    "Búsqueda binaria del producto",
    busqueda["producto"],
    "posición",
    busqueda["posicion"],
    "| B+Tree en rango",
    tuple(round(x, 2) for x in busqueda["rango"]),
    "->",
    busqueda["en_rango"],
    "productos",
)
print("KD-Tree contra fuerza bruta:", "CORRECTA" if resultado["knn_ok"] else "REVISAR")
print("Top PageRank de co-compra:", resultado["resumen_grafo"]["pagerank"][:5])
print("Top por grado:", resultado["resumen_grafo"]["grado"][:5])
print("Comunidades Louvain:", resultado["resumen_grafo"]["comunidades"], "| modularidad:", resultado["resumen_grafo"]["modularidad"])


E00059989 | view | producto 5301260 | consultar
0 Ingesta        | deque                            | E00059989 -> encolado en el stream vivo
1 Limpieza       | merge sort + dedup               | E00059989 -> conservado y ordenado por tiempo
2 Prioridad      | min-heap                         | Vista -> prioridad 6
3 Almacén        | hash + LRU                       | 5301260 -> HASH
4 Índice         | B+Tree                           | precio -> el árbol se actualiza al ver el producto
5 KNN exacto     | KD-Tree                          | precio y popularidad -> línea base ya contrastada con fuerza bruta
6 Fraude         | Bloom + Count-Min                | view -> omitida: no es un pago
7 Recomendación  | híbrido 1+4 / hnswlib            | unknown vitek -> 12701967,1005115,3800548,5300125,5300157
8 Co-compra      | grafo de orden                   | view -> omitida: no es una compra confirmada
9 Servicio       | Trie + panel                     | unknown -> 10701101,26800034,26800033

## Dashboard: Motor Inteligente de E-Commerce

Es el dashboard de la v1, ahora conectado al pipeline. Tiene seis pestañas:

1. **Buscar producto:** por ID (hash y caché LRU) o por texto (autocompletado con Trie). Ejemplos: `samsung`, `apple`, `electronics.smartphone`.
2. **Precios:** productos en un rango, con el B+Tree.
3. **Recomendar:** cinco productos para un ID, con el híbrido 1+4 (grafo de sesión y HNSW).
4. **Simular evento:** eliges vista o compra y un producto, y ves por qué ramas pasa. Si compras dos productos en la misma sesión de demo, el grafo de co-compra suma una arista.
5. **Panel y alertas:** ventas, consultas, alertas y bloqueos del stream.
6. **Grafo:** productos comprados juntos, más PageRank y comunidades.


In [10]:
def procesar_evento(resultado, producto, tipo, usuario=999_999_999, sesion="demo-sesion", marcar_sospechoso=False):
    almacen = resultado["almacen"]
    base = almacen.catalogo.get(int(producto))
    if base is None:
        return None, [("3 Almacén", "hash", str(producto), "producto no está en el catálogo")]
    contador = resultado.setdefault("contador_demo", 0) + 1
    resultado["contador_demo"] = contador
    ahora = pd.Timestamp.now(tz="UTC")
    dispositivo = f"dev-demo-{usuario}"
    if marcar_sospechoso:
        resultado["fraude"]["bloom"].agregar(dispositivo)
    if tipo == "purchase":
        prioridad, etiqueta = (2, "Express") if not marcar_sospechoso else (1, "Fraude")
    elif tipo == "cart":
        prioridad, etiqueta = 5, "Carrito"
    else:
        prioridad, etiqueta = 6, "Vista"
    evento = pd.Series({
        "evento_id": f"DEMO{contador:04d}",
        "event_time": ahora,
        "event_type": tipo,
        "product_id": int(producto),
        "category_code": base["categoria"],
        "brand": base["marca"],
        "price": base["precio"],
        "user_id": int(usuario),
        "user_session": sesion,
        "disponible": base["disponible"],
        "device_id": dispositivo,
        "payment_id": f"pay-demo-{usuario}",
        "fraude_confirmado": False,
        "hora": str(ahora.floor("h")),
    })
    cola = deque([evento])
    actual = cola.popleft()
    heap = []
    heapq.heappush(heap, (prioridad, 0, actual["evento_id"]))
    heapq.heappop(heap)
    item, origen = almacen.actualizar(actual)
    pasos = [
        ("0 Ingesta", "deque", actual["evento_id"], "recibido y sacado de la cola"),
        ("1 Limpieza", "validación", tipo, "campos completos"),
        ("2 Prioridad", "min-heap", etiqueta, f"prioridad {prioridad}"),
        ("3 Almacén", "hash + LRU", str(producto), f"{origen} | precio {base['precio']} | {base['categoria']} {base['marca']}"),
    ]
    sesiones_demo = resultado.setdefault("sesiones_demo", {})
    vistos = sesiones_demo.setdefault(sesion, [])
    salida = {"evento_id": actual["evento_id"], "tipo": tipo, "decision": "", "recomendaciones": []}
    if tipo in ("view", "cart"):
        historia = vistos + [int(producto)]
        recomendados = hibrido(historia, resultado["grafo_sesion"], resultado["modelo"])
        salida["decision"] = "consultar"
        salida["recomendaciones"] = recomendados
        pasos.append(("6 Fraude", "Bloom + Count-Min", tipo, "omitida: una vista no es un pago"))
        pasos.append(("7 Recomendación", "híbrido 1+4", ",".join(map(str, historia)), ",".join(map(str, recomendados)) or "sin candidatos"))
        pasos.append(("8 Co-compra", "grafo de orden", tipo, "omitida: no es una compra"))
    else:
        decision, motivos = decidir_fraude(actual, resultado["fraude"]["bloom"], resultado["fraude"]["cms"], resultado["fraude"]["exactos"])
        salida["decision"] = decision
        pasos.append(("6 Fraude", "Bloom + Count-Min + exacto", ",".join(motivos) or "sin señal", decision))
        pasos.append(("7 Recomendación", "híbrido", tipo, "omitida: una compra no pide sugerencias"))
        if decision == "bloquear":
            pasos.append(("8 Co-compra", "grafo de orden", str(producto), "no entra: compra bloqueada"))
        else:
            indice = -contador
            nuevas = unir_compra(resultado["cocompra"], indice, int(producto), sesion, ahora)
            pasos.append(("8 Co-compra", "grafo de orden", str(producto), f"entra al grafo | aristas nuevas {nuevas}"))
            resultado["panel"]["ventas"] += 1
        if decision in ("revisar", "bloquear"):
            resultado["panel"]["alertas"] += 1
            resultado["panel"]["bloqueos" if decision == "bloquear" else "revisiones"] += 1
            resultado.setdefault("alertas_demo", []).append((actual["evento_id"], int(producto), decision, ",".join(motivos)))
    if tipo in ("view", "cart"):
        resultado["panel"]["consultas"] += 1
    resultado["panel"]["eventos_stream"] += 1
    if int(producto) not in vistos:
        vistos.append(int(producto))
    pasos.append(("9 Servicio", "panel", actual["evento_id"], "panel actualizado"))
    return salida, pasos


def buscar_texto(resultado, texto, limite=10):
    hallados = []
    for producto in resultado["trie"].completar(texto.strip().lower(), limite=limite * 4):
        if producto not in hallados:
            hallados.append(producto)
        if len(hallados) >= limite:
            break
    return hallados


def tabla_productos(resultado, ids):
    catalogo = resultado["almacen"].catalogo
    filas = []
    for producto in ids:
        item = catalogo.get(int(producto))
        if item is None:
            continue
        filas.append({
            "producto": int(producto),
            "categoría": item["categoria"],
            "marca": item["marca"],
            "precio": round(item["precio"], 2),
            "disponible": "sí" if item["disponible"] else "no",
            "interacciones": item["popularidad"],
        })
    return pd.DataFrame(filas)


def ejemplo_producto(resultado):
    if resultado["modelo"]["populares"]:
        return int(resultado["modelo"]["populares"][0])
    return int(next(iter(resultado["almacen"].catalogo)))


In [11]:
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except Exception:
    pass

try:
    import ipywidgets as widgets
    from IPython.display import clear_output, display
    WIDGETS = True
except ImportError:
    WIDGETS = False
    print("No está ipywidgets. Instala con: pip install ipywidgets")

if WIDGETS:
    producto_demo = ejemplo_producto(resultado)
    precios_catalogo = [item["precio"] for item in resultado["almacen"].catalogo.values()]
    precio_bajo = float(np.percentile(precios_catalogo, 40)) if precios_catalogo else 0.0
    precio_alto = float(np.percentile(precios_catalogo, 60)) if precios_catalogo else 0.0

    titulo = widgets.HTML(
        "<h2>Proyecto EDA | Motor Inteligente de E-Commerce v2</h2>"
        f"<p>Dataset REES46, {DATASET_FILE}, {len(resultado['df']):,} eventos limpios, "
        f"{len(resultado['almacen'].catalogo):,} productos. Modo {RUN_MODE}.</p><hr>"
    )

    # 1. Buscar producto
    entrada_id = widgets.IntText(value=producto_demo, description="Producto ID:", style={"description_width": "initial"})
    boton_id = widgets.Button(description="Buscar por ID", button_style="primary")
    entrada_texto = widgets.Text(value="samsung", description="Texto:", placeholder="samsung, apple, electronics...", style={"description_width": "initial"})
    boton_texto = widgets.Button(description="Autocompletar", button_style="primary")
    salida_busqueda = widgets.Output()

    def buscar_id(_):
        with salida_busqueda:
            clear_output(wait=True)
            t0 = time.perf_counter()
            item, origen = resultado["almacen"].consultar(int(entrada_id.value))
            ms = (time.perf_counter() - t0) * 1000
            if item is None:
                print("Producto no encontrado.")
                return
            print(f"Origen: {origen} | tiempo {ms:.4f} ms (repite la búsqueda para ver la caché LRU)")
            display(tabla_productos(resultado, [int(entrada_id.value)]))

    def buscar_por_texto(_):
        with salida_busqueda:
            clear_output(wait=True)
            t0 = time.perf_counter()
            ids = buscar_texto(resultado, entrada_texto.value)
            ms = (time.perf_counter() - t0) * 1000
            print(f"Trie | '{entrada_texto.value}' | {len(ids)} resultados | {ms:.4f} ms")
            if ids:
                display(tabla_productos(resultado, ids))
            else:
                print("Sin coincidencias. Prueba con una marca o una categoría que empiece igual.")

    boton_id.on_click(buscar_id)
    boton_texto.on_click(buscar_por_texto)
    panel_busqueda = widgets.VBox([
        widgets.HTML("<h3>Buscar producto</h3>"),
        widgets.HBox([entrada_id, boton_id]),
        widgets.HBox([entrada_texto, boton_texto]),
        salida_busqueda,
    ])

    # 2. Precios
    precio_min = widgets.FloatText(value=round(precio_bajo, 2), description="Mínimo:")
    precio_max = widgets.FloatText(value=round(precio_alto, 2), description="Máximo:")
    boton_precio = widgets.Button(description="Buscar por precio", button_style="primary")
    salida_precio = widgets.Output()

    def buscar_precio(_):
        with salida_precio:
            clear_output(wait=True)
            if precio_min.value > precio_max.value:
                print("El mínimo no puede superar al máximo.")
                return
            t0 = time.perf_counter()
            ids = resultado["almacen"].precios.buscar_rango(precio_min.value, precio_max.value)
            ms = (time.perf_counter() - t0) * 1000
            print(f"B+Tree | {len(ids)} productos | {ms:.4f} ms | se muestran 20")
            display(tabla_productos(resultado, ids[:20]))

    boton_precio.on_click(buscar_precio)
    panel_precio = widgets.VBox([
        widgets.HTML("<h3>Búsqueda por rango de precios</h3>"),
        precio_min, precio_max, boton_precio, salida_precio,
    ])

    # 3. Recomendar
    entrada_rec = widgets.IntText(value=producto_demo, description="Producto ID:", style={"description_width": "initial"})
    boton_rec = widgets.Button(description="Recomendar 5", button_style="primary")
    salida_rec = widgets.Output()

    def recomendar(_):
        with salida_rec:
            clear_output(wait=True)
            producto = int(entrada_rec.value)
            if producto not in resultado["almacen"].catalogo:
                print("Producto no encontrado.")
                return
            t0 = time.perf_counter()
            ids = hibrido([producto], resultado["grafo_sesion"], resultado["modelo"])
            ms = (time.perf_counter() - t0) * 1000
            print(f"Híbrido 1+4 ({resultado['modelo']['indice'].backend} + grafo de sesión) | {ms:.4f} ms")
            print("Producto de referencia:")
            display(tabla_productos(resultado, [producto]))
            print("Recomendados:")
            display(tabla_productos(resultado, ids))

    boton_rec.on_click(recomendar)
    panel_rec = widgets.VBox([
        widgets.HTML("<h3>Recomendaciones</h3>"),
        entrada_rec, boton_rec, salida_rec,
    ])

    # 4. Simular evento
    tipo_evento = widgets.Dropdown(
        options=[("Vista", "view"), ("Carrito", "cart"), ("Compra", "purchase")],
        value="view",
        description="Evento:",
    )
    entrada_sim = widgets.IntText(value=producto_demo, description="Producto ID:", style={"description_width": "initial"})
    entrada_sesion = widgets.Text(value="demo-sesion-1", description="Sesión:")
    sospechoso = widgets.Checkbox(value=False, description="Dispositivo señalado (Bloom)")
    boton_sim = widgets.Button(description="Enviar al pipeline", button_style="success")
    salida_sim = widgets.Output()

    def simular(_):
        with salida_sim:
            clear_output(wait=True)
            salida, pasos = procesar_evento(
                resultado,
                int(entrada_sim.value),
                tipo_evento.value,
                sesion=entrada_sesion.value,
                marcar_sospechoso=sospechoso.value,
            )
            display(pd.DataFrame(pasos, columns=["etapa", "estructura", "entrada", "salida"]))
            if salida and salida["recomendaciones"]:
                print("Recomendados:")
                display(tabla_productos(resultado, salida["recomendaciones"]))

    boton_sim.on_click(simular)
    panel_sim = widgets.VBox([
        widgets.HTML(
            "<h3>Simular un evento</h3>"
            "<p>Una vista recomienda. Una compra se revisa por fraude y, si no se bloquea, "
            "entra al grafo. Compra dos productos con la misma sesión para crear una arista.</p>"
        ),
        tipo_evento, entrada_sim, entrada_sesion, sospechoso, boton_sim, salida_sim,
    ])

    # 5. Panel y alertas
    boton_panel = widgets.Button(description="Actualizar panel", button_style="primary")
    salida_panel = widgets.Output()

    def mostrar_panel(_):
        with salida_panel:
            clear_output(wait=True)
            display(pd.DataFrame([resultado["panel"]]))
            populares = [p for p, _ in Counter(
                {p: item["popularidad"] for p, item in resultado["almacen"].catalogo.items()}
            ).most_common(10)]
            print("Top 10 productos por interacciones:")
            display(tabla_productos(resultado, populares))
            alertas = [
                (t["evento_id"], t["producto"], t["decision"], t["motivos"])
                for t in resultado["trazas"]
                if t["decision"] in ("revisar", "bloquear")
            ][:20] + resultado.get("alertas_demo", [])
            print("Alertas de fraude (primeras 20 del stream y las de la demo):")
            if alertas:
                display(pd.DataFrame(alertas, columns=["evento", "producto", "decisión", "motivos"]))
            else:
                print("Sin alertas.")

    boton_panel.on_click(mostrar_panel)
    panel_ventas = widgets.VBox([
        widgets.HTML("<h3>Panel de ventas, actividad y alertas</h3>"),
        boton_panel, salida_panel,
    ])

    # 6. Grafo
    boton_grafo = widgets.Button(description="Ver grafo de co-compra", button_style="primary")
    salida_grafo = widgets.Output()

    def mostrar_grafo(_):
        with salida_grafo:
            clear_output(wait=True)
            grafo = resultado["cocompra"]["grafo"]
            if grafo is None:
                print("No está networkx.")
                return
            resumen = resumen_grafo(resultado["cocompra"])
            print(f"Nodos {resumen['nodos']} | aristas {resumen['aristas']} | comunidades {resumen['comunidades']} | modularidad {resumen['modularidad']}")
            if resumen["pagerank"]:
                print("Top PageRank:")
                display(pd.DataFrame(resumen["pagerank"][:10], columns=["producto", "pagerank"]))
            conectados = [n for n in grafo.nodes if grafo.degree(n) > 0]
            if not conectados:
                print("Todavía no hay productos comprados juntos. En modo quick hay pocas compras por sesión; usa full o simula dos compras en la misma sesión.")
                return
            import matplotlib.pyplot as plt
            sub = grafo.subgraph(sorted(conectados, key=lambda n: grafo.degree(n, weight="weight"), reverse=True)[:60])
            plt.figure(figsize=(10, 6))
            import networkx as nx
            nx.draw_networkx(sub, pos=nx.spring_layout(sub, seed=SEED), with_labels=True, node_size=500, font_size=7)
            plt.title("Productos comprados en la misma sesión (hasta 60 nodos)")
            plt.axis("off")
            plt.show()

    boton_grafo.on_click(mostrar_grafo)
    panel_grafo = widgets.VBox([
        widgets.HTML("<h3>Grafo de co-compra</h3>"),
        boton_grafo, salida_grafo,
    ])

    # widgets.Tab no se dibuja en Colab: se usa un selector que cambia el panel visible.
    paneles = {
        "Buscar": panel_busqueda,
        "Precios": panel_precio,
        "Recomendar": panel_rec,
        "Simular evento": panel_sim,
        "Panel y alertas": panel_ventas,
        "Grafo": panel_grafo,
    }
    selector = widgets.ToggleButtons(options=list(paneles), value="Buscar")
    contenedor = widgets.VBox([paneles["Buscar"]])

    def cambiar_panel(cambio):
        contenedor.children = [paneles[cambio["new"]]]

    selector.observe(cambiar_panel, names="value")
    display(widgets.VBox([titulo, selector, contenedor]))


HTML(value='<h2>Proyecto EDA | Motor Inteligente de E-Commerce v2</h2><p>Dataset REES46, 2019-Oct.csv, 99,983 …

## Recomendación en el corte temporal

El modelo se ajusta solo con el 60 % inicial. La validación es el 10 % siguiente y la prueba el 30 % final. La cuota 1+4 se mantiene: un candidato del índice vectorial y cuatro del grafo de sesión. Si una ventana no tiene sesiones con historia, la tabla queda en cero y se dice explícitamente; no se reemplaza el grafo de co-compra por este grafo de vistas.


In [12]:
tabla_val, tabla_test = evaluar_recomendacion(resultado)
print("Validación")
print(tabla_val.to_string(index=False))
print("\nPrueba")
print(tabla_test.to_string(index=False))


Validación
         metodo  sesiones  hit_rate_5  precision_5  recall_5  latencia_mediana_ms
Grafo de sesión       300    0.213333     0.047333  0.139097             0.081515
           HNSW       300    0.106667     0.023333  0.066835             0.810140
    Híbrido 1+4       300    0.200000     0.044000  0.128402             0.959345

Prueba
         metodo  sesiones  hit_rate_5  precision_5  recall_5  latencia_mediana_ms
Grafo de sesión       300    0.313333     0.078667  0.160912             0.086212
           HNSW       300    0.146667     0.034667  0.073076             0.796688
    Híbrido 1+4       300    0.286667     0.072000  0.146863             0.944586


## Benchmarks

Cada tamaño sale del mismo lote leído, sin copiar el CSV. En `full`, 1M y 10M usan 3 repeticiones: medir 100 veces esas estructuras en Python puro no aporta una corrida terminable. El resto del modo `full` usa 100. El modo `quick` usa 7.


In [13]:
def medir(funcion, reps):
    tiempos = []
    for _ in range(reps):
        t0 = time.perf_counter()
        funcion()
        tiempos.append(time.perf_counter() - t0)
    return float(np.mean(tiempos)), float(np.std(tiempos))


def memoria_de(funcion):
    tracemalloc.start()
    funcion()
    _, pico = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return pico / (1024 ** 2)


def correr_benchmarks(resultado):
    df = resultado["df"]
    productos = df["product_id"].astype(int).to_numpy()
    filas = []
    plataforma = platform.platform()
    for n in TAMANOS_BENCH:
        if n > len(df):
            continue
        reps = repeticiones_para(n)
        muestra = productos[:n]
        objetivo = -1

        def lineal(muestra=muestra, objetivo=objetivo):
            return objetivo in muestra

        tabla = set(muestra.tolist())

        def por_hash(tabla=tabla, objetivo=objetivo):
            return objetivo in tabla

        media, desv = medir(lineal, reps)
        filas.append(("busqueda_lineal", n, media, desv, 0.0, "encontrado", 0, reps, SEED, plataforma))
        media, desv = medir(por_hash, reps)
        mem = memoria_de(lambda muestra=muestra: set(muestra.tolist()))
        filas.append(("hash", n, media, desv, mem, "encontrado", 0, reps, SEED, plataforma))

        base = muestra.tolist()
        rng_local = random.Random(SEED)
        rng_local.shuffle(base)

        def ordenar(base=base):
            merge_sort(base, lambda x: x)

        media, desv = medir(ordenar, reps)
        filas.append(("merge_sort", n, media, desv, 0.0, "n", n, reps, SEED, plataforma))

        def heap_n(base=base):
            heap = []
            for i, valor in enumerate(base):
                heapq.heappush(heap, (int(valor), i))
            while heap:
                heapq.heappop(heap)

        media, desv = medir(heap_n, reps)
        filas.append(("heap", n, media, desv, 0.0, "n", n, reps, SEED, plataforma))

        insertados = set(int(x) for x in muestra[::10])
        bloom = BloomFilter(max(len(insertados), 1), 0.01)
        for valor in insertados:
            bloom.agregar(valor)
        consultas = [int(x) for x in muestra[: min(n, 20_000)]]
        fp = sum(1 for valor in consultas if bloom.contiene(valor) and valor not in insertados)
        consultas_neg = [valor for valor in consultas if valor not in insertados]

        def consultar_bloom(bloom=bloom, consultas=consultas):
            for valor in consultas:
                bloom.contiene(valor)

        media, desv = medir(consultar_bloom, reps)
        def construir_bloom(insertados=insertados):
            filtro = BloomFilter(max(len(insertados), 1), 0.01)
            for valor in insertados:
                filtro.agregar(valor)
            return filtro

        mem = memoria_de(construir_bloom)
        tasa = fp / len(consultas_neg) if consultas_neg else 0.0
        filas.append(("bloom_vs_set", n, media, desv, mem, "falsos_positivos", tasa, reps, SEED, plataforma))

        cms = CountMinSketch()
        conteo = Counter(int(x) for x in muestra)
        for valor in muestra:
            cms.agregar(int(valor))
        claves = list(conteo)[:100]
        error = float(np.mean([abs(cms.estimar(clave) - conteo[clave]) for clave in claves])) if claves else 0.0

        def consultar_cms(cms=cms, claves=claves):
            for clave in claves:
                cms.estimar(clave)

        media, desv = medir(consultar_cms, reps)
        filas.append(("count_min_vs_exacto", n, media, desv, cms.tabla.nbytes / (1024 ** 2), "mae", error, reps, SEED, plataforma))

    filas.extend(benchmark_similitud(resultado))
    filas.extend(benchmark_comunidades(resultado))
    return pd.DataFrame(
        filas,
        columns=["estructura", "n", "tiempo_promedio_s", "desviacion_s", "memoria_mb", "metrica", "valor", "repeticiones", "semilla", "plataforma"],
    )


class MinHashLSH:
    def __init__(self, num_perm=32, bandas=8):
        self.num_perm = num_perm
        self.filas = num_perm // bandas
        self.bandas = bandas
        self.buckets = [defaultdict(list) for _ in range(bandas)]
        self.firmas = {}
        self.conjuntos = {}

    def _firma(self, items):
        firma = []
        recorte = list(items)[:200]
        for i in range(self.num_perm):
            menor = None
            for item in recorte:
                digest = hashlib.sha256(f"{i}|{item}".encode()).digest()
                valor = int.from_bytes(digest[:4], "little")
                menor = valor if menor is None else min(menor, valor)
            firma.append(0 if menor is None else menor)
        return firma

    def agregar(self, clave, items):
        if not items:
            return
        firma = self._firma(items)
        self.firmas[clave] = firma
        self.conjuntos[clave] = set(items)
        for b in range(self.bandas):
            inicio = b * self.filas
            self.buckets[b][tuple(firma[inicio:inicio + self.filas])].append(clave)

    def candidatos(self, clave):
        firma = self.firmas[clave]
        hallados = set()
        for b in range(self.bandas):
            inicio = b * self.filas
            hallados.update(self.buckets[b][tuple(firma[inicio:inicio + self.filas])])
        hallados.discard(clave)
        return hallados


def jaccard(a, b):
    if not a and not b:
        return 0.0
    inter = len(a & b)
    return inter / (len(a) + len(b) - inter)


def benchmark_similitud(resultado):
    usuarios = defaultdict(set)
    df = resultado["df"].iloc[: resultado["corte_train"]]
    for producto, usuario in zip(df["product_id"], df["user_id"]):
        usuarios[int(producto)].add(int(usuario))
        if len(usuarios) > 250 and len(usuarios[int(producto)]) > 30:
            continue
    pares = sorted(usuarios.items(), key=lambda item: len(item[1]), reverse=True)[:120]
    pares = [(p, s) for p, s in pares if len(s) >= 2]
    if len(pares) < 5:
        return [("minhash_lsh", len(resultado["df"]), 0.0, 0.0, 0.0, "recall", 0.0, 1, SEED, platform.platform())]
    lsh = MinHashLSH()
    for producto, conjunto in pares:
        lsh.agregar(producto, conjunto)
    consulta = pares[0][0]
    exactos = []
    for producto, conjunto in pares[1:]:
        sim = jaccard(lsh.conjuntos[consulta], conjunto)
        if sim >= 0.05:
            exactos.append(producto)
    t0 = time.perf_counter()
    candidatos = lsh.candidatos(consulta)
    tiempo = time.perf_counter() - t0
    recall = (len(set(exactos) & candidatos) / len(exactos)) if exactos else 1.0
    vectores = resultado["modelo"]["vectores"]
    ids = resultado["modelo"]["ids"]
    recall_h = 1.0
    tiempo_h = 0.0
    if len(ids) >= 5:
        k = min(6, len(ids))
        t0 = time.perf_counter()
        aprox, _ = resultado["modelo"]["indice"].buscar(vectores[0], k)
        tiempo_h = time.perf_counter() - t0
        orden, _ = knn_exacto(vectores, vectores[0], k)
        exacto_ids = [ids[i] for i in orden if ids[i] != ids[0]]
        aprox = [p for p in aprox if p != ids[0]]
        recall_h = len(set(exacto_ids) & set(aprox)) / len(exacto_ids) if exacto_ids else 1.0
    plataforma = platform.platform()
    return [
        ("minhash_vs_jaccard", len(pares), tiempo, 0.0, 0.0, "recall", recall, 1, SEED, plataforma),
        ("hnsw_vs_knn", len(ids), tiempo_h, 0.0, vectores.nbytes / (1024 ** 2), "recall_at_k", recall_h, 1, SEED, plataforma),
    ]


def benchmark_comunidades(resultado):
    resumen = resultado["resumen_grafo"]
    plataforma = platform.platform()
    if resumen["aristas"] == 0:
        return [("louvain_vs_grado", resumen["nodos"], 0.0, 0.0, 0.0, "aristas", 0, 1, SEED, plataforma)]
    top_pr = [p for p, _ in resumen["pagerank"][:10]]
    top_gr = [p for p, _ in resumen["grado"][:10]]
    coinciden = len(set(top_pr) & set(top_gr)) / 10
    return [(
        "pagerank_vs_grado",
        resumen["nodos"],
        0.0,
        0.0,
        0.0,
        "solape_top10",
        coinciden,
        1,
        SEED,
        plataforma + f" | comunidades {resumen['comunidades']} | modularidad {resumen['modularidad']}",
    )]


benchmarks = correr_benchmarks(resultado)
print(benchmarks.to_string(index=False))
if RUN_MODE == "full":
    print("1M y 10M usan 3 repeticiones para que la medición termine. El motivo es el costo del código Python puro, no un cambio de algoritmo.")


         estructura     n  tiempo_promedio_s  desviacion_s  memoria_mb          metrica        valor  repeticiones  semilla                                                                              plataforma
    busqueda_lineal  1000       2.950857e-05  4.133301e-05    0.000000       encontrado     0.000000             7       42                                                    Linux-6.6.122+-x86_64-with-glibc2.39
               hash  1000       5.370000e-07  3.891522e-07    0.077461       encontrado     0.000000             7       42                                                    Linux-6.6.122+-x86_64-with-glibc2.39
         merge_sort  1000       2.466512e-03  4.848459e-05    0.000000                n  1000.000000             7       42                                                    Linux-6.6.122+-x86_64-with-glibc2.39
               heap  1000       7.095209e-04  8.770771e-05    0.000000                n  1000.000000             7       42                             

In [14]:
bloqueos = [t for t in resultado["trazas"] if t["decision"] == "bloquear"]
permitidas = [t for t in resultado["trazas"] if t["tipo"] == "purchase" and t["decision"] == "permitir"]
vistas = [t for t in resultado["trazas"] if t["tipo"] == "view"]
validacion = []
validacion.append(("orden temporal", bool(resultado["df"]["event_time"].is_monotonic_increasing)))
validacion.append(("vista no se bloquea", all(t["decision"] == "consultar" for t in vistas) if vistas else True))
validacion.append(("bloqueo no actualiza co-compra", all(not t["actualizo_cocompra"] for t in bloqueos) if bloqueos else True))
validacion.append(("compra permitida entra al grafo", all(t["actualizo_cocompra"] for t in permitidas) if permitidas else True))
validacion.append(("búsqueda binaria", resultado["busqueda"]["posicion"] >= 0))
validacion.append(("kd-tree", bool(resultado["knn_ok"])))
validacion.append(("vector fijo", resultado["modelo"]["vectores"].shape[1] == VECTOR_DIM if len(resultado["modelo"]["ids"]) else True))
muestra_idx = list(range(len(resultado["df"])))
tiempos_idx = resultado["df"]["event_time"].astype("int64").to_numpy()
random.Random(SEED).shuffle(muestra_idx)
ordenado_ext = ordenar_indices(muestra_idx, tiempos_idx, tamano_bloque=50)
validacion.append((
    "merge sort externo",
    ordenado_ext == sorted(muestra_idx, key=lambda i: (int(tiempos_idx[i]), i)),
))
print("Validación del contrato")
for nombre, ok in validacion:
    print(("OK  " if ok else "FALLO") + " | " + nombre)
print("Listo." if all(ok for _, ok in validacion) else "Hay una validación pendiente.")


Validación del contrato
OK   | orden temporal
OK   | vista no se bloquea
OK   | bloqueo no actualiza co-compra
OK   | compra permitida entra al grafo
OK   | búsqueda binaria
OK   | kd-tree
OK   | vector fijo
OK   | merge sort externo
Listo.
